# 3 · Rules over collections

**The problem.** The latest reading isn't the whole story. A logger records the temperature every few minutes, and a
shipment that spent an hour at 12 °C may be back at 5 °C by the time it's checked. The real rules are about *all* the
readings: every one in range, or at most two brief excursions, never above 12 °C. And about the loggers a shipment
carries: each one's battery must last the trip.

This case study writes rules over lists and over an object's related objects, with **quantifiers**.

In [1]:
from mbse.Expressions import Evaluators, Expressions as E
from mbse.Expressions.Dialects.Python import Text
from toolkit import cold_chain, show

store = cold_chain()
steady = store.Shipment().id("S1").readings([4.0, 5.5, 6.1, 5.0]).create()
blip = store.Shipment().id("S2").readings([4.0, 8.6, 9.1, 5.0]).create()
cooked = store.Shipment().id("S3").readings([4.0, 12.5, 14.0, 13.2]).create()
silent = store.Shipment().id("S4").create()  # no readings uploaded
shipments = [steady, blip, cooked, silent]
this, c = E.variable("this"), E.variable("c")


def table(rule):
    return {shipment.id: Evaluators.predicate(rule, shipment) for shipment in shipments}

## Step 1: a list is a collection

`readings` is a list property. Reading it gives a **collection**, which a handful of operations take: `count`, `item`
(by position, from 0), `in`, `sum`, `min`, `max` and `unique`.

In [2]:
for rule in [this.readings, this.readings.count(), this.readings.item(0), this.readings.max(),
             E.literal(9.1).in_(this.readings), this.readings.unique()]:
    print(f"{show(rule):40}", Evaluators.OfAny(rule, {"this": blip}))

get(this, 'readings')                    [4.0, 8.6, 9.1, 5.0]
count(get(this, 'readings'))             4
item(get(this, 'readings'), 0)           4.0
max(get(this, 'readings'))               9.1
in(9.1, get(this, 'readings'))           True
unique(get(this, 'readings'))            True


Here they're built with writers; `FromFunction` also reads them from Python's own spellings, `len(xs)`, `xs[0]`,
`max(xs)` and `9.1 in xs`, and the next steps use those.

Collections follow the same rules about unknowns as everything else. There's no item at position 9, so it's unknown,
not an error; the silent shipment has no readings at all, so everything about them is unknown:

In [3]:
for rule in [this.readings.item(9), this.readings.count(), this.readings.max()]:
    print(f"{show(rule):40}", Evaluators.OfAny(rule, {"this": blip}), Evaluators.OfAny(rule, {"this": silent}))

item(get(this, 'readings'), 9)           None None
count(get(this, 'readings'))             4 None
max(get(this, 'readings'))               9.1 None


## Step 2: quantifiers

A **quantifier** binds a name to each item in turn, here `c`, and tests a body. `all` holds when the body holds for
every item, `any` when it holds for one, and `count` counts the items it holds for. In a lambda, they're Python's
`all(...)` and `any(...)` of a generator expression, and `sum(1 for ... if ...)`; writers spell them `.all`, `.any`
and `.count_where`:

In [4]:
every_in_range = Text.FromFunction(lambda this: all(2.0 <= c <= 8.0 for c in this.readings))
print(show(every_in_range))
table(every_in_range)

all(get(this, 'readings'), c => and(le(2.0, c), le(c, 8.0)))


{'S1': True, 'S2': False, 'S3': False, 'S4': None}

The steady shipment passes and the others fail; the silent one is unknown. The quantifier is a term of its own
kind, which binds `c` the way a let does, so `c` means nothing outside the body.

The real rule allows brief excursions: at most two readings above 8 °C, and none above 12 °C. That's a count and a
maximum:

In [5]:
excursions = Text.FromFunction(lambda this: sum(1 for c in this.readings if c > 8.0))
allowed = Text.FromFunction(lambda this: excursions <= 2 and max(this.readings) <= 12.0)
print(show(allowed))
table(allowed)

and(le(count(get(this, 'readings'), c => gt(c, 8.0)), 2), le(max(get(this, 'readings')), 12.0))


{'S1': True, 'S2': True, 'S3': False, 'S4': None}

The blip is allowed now; the cooked shipment still isn't. `allowed` uses `excursions`, an expression from the
surrounding code, as it is: a lambda can build on rules already written.

## Step 3: empty collections

An empty collection is data too, and quantifiers answer the way logic does: every item of nothing satisfies anything
(`all` is true), none satisfies something (`any` is false), and the count is 0. What has no answer is the greatest of
nothing, so `max` is unknown, and a rule that relies on it says so:

In [6]:
empty = store.Shipment().id("S5").readings([]).create()
for name, rule in [("every_in_range", every_in_range), ("any above 8", this.readings.any("c", c.gt(8.0))),
                   ("excursions", excursions), ("max", this.readings.max()), ("allowed", allowed)]:
    print(f"{name:16}", Evaluators.OfAny(rule, {"this": empty}))

every_in_range   True
any above 8      False
excursions       0
max              None
allowed          None


`allowed` for an empty list is unknown, not true: `excursions` is 0, which is fine, but the maximum is unknown. A
shipment that logged nothing hasn't shown it stayed cold, and the rule doesn't pretend otherwise. If an empty log
*should* pass, say so explicitly, with `count`:

In [7]:
lenient = Text.FromFunction(lambda this: len(this.readings) == 0 or allowed)
print(Evaluators.predicate(lenient, empty), Evaluators.predicate(lenient, cooked))

True False


## Step 4: an object's related objects

A shipment carries loggers, through the relation `Carries`, whose entries record the slot each logger rides in.
`entries(object, name)` gives an object's entries in a relation as a collection of **records**, which `get` reads like
objects: the linked logger, and the entry's own properties.

In [8]:
trip = (store.Shipment().id("S6").readings([5.0])
        .loggers(lambda e: e.slot(1).logger(lambda l: l.serial("L-17").battery(80)))
        .loggers(lambda e: e.slot(2).logger(lambda l: l.serial("L-22").battery(15))).create())
e = E.variable("e")
for rule in [this.entries("loggers").count(), this.entries("loggers").item(1).get("slot"),
             this.entries("loggers").item(1).get("logger").get("serial")]:
    print(f"{show(rule):70}", Evaluators.OfAny(rule, {"this": trip}))

count(entries(this, 'loggers'))                                        2
get(item(entries(this, 'loggers'), 1), 'slot')                         2
get(get(item(entries(this, 'loggers'), 1), 'logger'), 'serial')        L-22


`.get("logger")` is the writer of `get(e, 'logger')`; `e.logger` would be the same. Entries have no Python spelling,
so rules over them are written with writers. "Every logger's battery lasts the trip" is a quantifier over the
entries:

In [9]:
charged = this.entries("loggers").all("e", e.logger.battery.ge(20))
print(show(charged))
print(Evaluators.predicate(charged, trip))
Evaluators.OfAny(this.entries("loggers").count_where("e", e.logger.battery.lt(20)), {"this": trip})

all(entries(this, 'loggers'), e => ge(get(get(e, 'logger'), 'battery'), 20))
False


1

## Why it works this way

- **Quantifiers instead of loops.** A loop can do anything, including never finish. A quantifier visits each item of
  data once, so a rule with quantifiers is still bounded, total and pure, and still a constraint a solver or a hardware
  checker could take.
- **Collections come from data.** There are no list literals: a collection is a list property, or an object's entries,
  or a variable the caller binds. That keeps rules about the data, not little programs of their own.
- **The empty case is honest.** `all` of nothing is true because nothing violates it; the maximum of nothing doesn't
  exist. Rules that need an answer for the empty case say what it is.